# CS4100 Final Project  
Team Members: Khushi Khan, Dustin Zhang, Kayla Handley, Koena Gupta

In [23]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchvision
import torchmetrics
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split    
from torch import nn
from torch import optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [24]:
mp3_df = pd.read_csv('../data/cleaned/fma_cleaned_dataset_emotion_labels.csv', low_memory=False)

mp3_df.head()

,track_id,title,genre_top,mp3_path,valence,energy,emotion_joy_excitement,emotion_peaceful_content,emotion_anger_tension,emotion_sadness,emotion_joy_excitement_softmax,emotion_peaceful_content_softmax,emotion_anger_tension_softmax,emotion_sadness_softmax
0,2,Food,Hip-Hop,../data/raw/fma_small/000/000002.mp3,0.576661,0.634476,0.606258,0.482776,0.539340,0.395489,0.275553,0.243544,0.257717,0.223187
1,5,This World,Hip-Hop,../data/raw/fma_small/000/000005.mp3,0.621661,0.701470,0.662768,0.487639,0.563560,0.340779,0.288217,0.241914,0.260996,0.208873
2,10,Freeway,Pop,../data/raw/fma_small/000/000010.mp3,0.963590,0.924525,0.944260,0.683448,0.654245,0.059254,0.341135,0.262819,0.255255,0.140791
3,140,Queen Of The Wires,Folk,../data/raw/fma_small/000/000140.mp3,0.609991,0.265685,0.470467,0.675022,0.333688,0.587931,0.236755,0.290493,0.206489,0.266264
4,141,Ohio,Folk,../data/raw/fma_small/000/000141.mp3,0.163950,0.075632,0.127671,0.663828,0.593591,0.881316,0.155578,0.265949,0.247911,0.330562


In [25]:
# Targets to predict
y = mp3_df[['valence', 'energy']].to_numpy()
n_outputs = y.shape[1]

# Features to pass in
X = mp3_df.drop(columns=['valence', 'energy']).to_numpy()
n_samples, n_features = X.shape

In [26]:
# Defining a custom dataset class
class SpectrogramDataset(Dataset):
    def __init__(self, df, transform=None, resize=None):
        self.df = df
        self.transform = transform
        self.resize = resize
        self.images = {img["track_id"]: img for img in df}
        self.image_ids = list(self.images.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]

        pass

In [27]:
# Splitting data into train and test sets
X_train, y_train, X_test, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

emotion_classes = ('emotion_joy_excitement', 'emotion_peaceful_content', 'emotion_anger_tension', 'emotion_sadness')

In [28]:
# Initialize the dataset
img_dims = (256, 256)
transform = transforms.ToTensor()
train_ds = SpectrogramDataset(X_train, transform=transform, resize=img_dims)
test_ds = SpectrogramDataset(X_test, transform=transform, resize=img_dims)

# Initialize the data loaders
train_dataloader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
test_dataloader = DataLoader(test_ds, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
# # Utility function to visualize an image
# def imshow(img):
#    npimg = img.numpy()
#    plt.imshow(np.transpose(npimg, (1, 2, 0)))
#    plt.show()

# imshow(...)

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(CNN, self).__init__()

        # 1st convolutional layer
        self.conv1 = nn.Conv2d(
            in_channels=in_channels, 
            out_channels=6,
            kernel_size=3,
            padding=1)
        
        # Max pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 2nd convolutional layer
        self.conv2 = nn.Conv2d(
            in_channels=6, 
            out_channels=16, 
            kernel_size=3,
            padding=1)

        # Fully connected layer
        self.fc1 = None

    def forward(self, x):
        x = F.relu(self.conv1(x))  # Apply first convolution and ReLU activation
        x = self.pool(x)           # Apply max pooling
        x = F.relu(self.conv2(x))  # Apply second convolution and ReLU activation
        x = self.pool(x)           # Apply max pooling
        x = x.view(x.size(0), -1)  # Flatten the tensor

        if self.fc1 is None:
            # Apply fully connected layer
            self.fc1 = nn.Linear(x.shape[1], 4).to(x.device)
        x = self.fc1(x)
        return x

device = "cuda" if torch.cuda.is_available() else "cpu"

# Since the spectrograms are grayscale, in-channels=1
model = CNN(in_channels=1, num_classes=len(emotion_classes)).to(device)

print(f"Model architecture:\n {model}")

Model architecture:
 CNN(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs=10
for epoch in range(num_epochs):
 # Iterate over training batches
   print(f"Epoch [{epoch + 1}/{num_epochs}]")

   for batch_index, (data, targets) in enumerate(tqdm(train_dataloader)):
       data = data.to(device)
       targets = targets.to(device)
       scores = model(data)
       loss = criterion(scores, targets)
       optimizer.zero_grad()
       loss.backward()
       optimizer.step()

Epoch [1/10]


  0%|          | 0/905 [00:00<?, ?it/s]


ValueError: too many values to unpack (expected 2)